# Bouldering Model Analysis — Bayesian (PyMC)

In [ ]:
import json
from pathlib import Path

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import expit as sigmoid

plt.rcParams.update({
    "figure.dpi": 200,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

In [ ]:
run_dirs = sorted(Path("runs").glob("bayesian_*"), key=lambda p: p.stat().st_mtime, reverse=True)
if not run_dirs:
    raise FileNotFoundError("No bayesian run directory found under runs/")
RUN_DIR = str(run_dirs[0])
print(f"Using run: {RUN_DIR}")

POSTERIOR_PATH = f"{RUN_DIR}/posterior.nc"
DATASET_PATH = f"{RUN_DIR}/dataset.pt"
BOULDERS_PATH = "data/boulders.jsonl"
ASCENTS_PATH = "data/ascents.jsonl"

In [ ]:
idata = az.from_netcdf(POSTERIOR_PATH)
print(idata.posterior)

In [ ]:
import torch
data = torch.load(DATASET_PATH, weights_only=True)

n_draws = idata.posterior.dims["draw"]

theta_samples = idata.posterior["theta"].values[0]
alpha_samples = idata.posterior["alpha"].values[0]
d_samples = idata.posterior["d"].values[0]
pi_samples = idata.posterior["pi"].values[0]
beta_samples = idata.posterior["beta"].values[0]
log_gamma_samples = idata.posterior["log_gamma"].values[0]
mu_samples = idata.posterior["mu"].values[0]
gamma_samples = np.exp(log_gamma_samples)

theta_mean = theta_samples.mean(axis=0)
theta_std  = theta_samples.std(axis=0)
alpha_mean = alpha_samples.mean(axis=0)
alpha_std  = alpha_samples.std(axis=0)
d_mean = d_samples.mean(axis=0)
d_std  = d_samples.std(axis=0)
pi_mean = pi_samples.mean(axis=0)
pi_std  = pi_samples.std(axis=0)
beta_mean = float(beta_samples.mean())
beta_std  = float(beta_samples.std())
gamma_mean = float(gamma_samples.mean())
mu_mean = float(mu_samples.mean())
mu_std  = float(mu_samples.std())

idx_to_boulder = {v: k for k, v in data["boulder_to_idx"].items()}
boulder_by_idx = {}
for bid, i in data["boulder_to_idx"].items():
    boulder_by_idx[i] = bid

bid_to_meta = {}
with open(BOULDERS_PATH) as f:
    for line in f:
        b = json.loads(line)
        bid_to_meta[b["boulder_id"]] = b

n_climbers = len(theta_mean)
n_boulders = len(d_mean)
print(f"Boulders: {n_boulders}, Climbers: {n_climbers}, Posterior draws: {n_draws}")

In [ ]:
def grade_to_numeric(grade: str) -> float | None:
    mapping = {
        "2": 2, "3A": 3, "3B": 4, "3C": 5,
        "4A": 6, "4B": 7, "4C": 8,
        "5A": 9, "5B": 10, "5C": 11,
        "6A": 12, "6A+": 13, "6B": 14, "6B+": 15, "6C": 16, "6C+": 17,
        "7A": 18, "7A+": 19, "7B": 20, "7B+": 21, "7C": 22, "7C+": 23,
        "8A": 24, "8A+": 25, "8B": 26, "8B+": 27, "8C": 28, "8C+": 29,
    }
    return mapping.get(grade)

def grade_label(n: float) -> str:
    labels = ["2", "3A", "3B", "3C", "4A", "4B", "4C",
              "5A", "5B", "5C", "6A", "6A+", "6B", "6B+", "6C", "6C+",
              "7A", "7A+", "7B", "7B+", "7C", "7C+",
              "8A", "8A+", "8B", "8B+", "8C", "8C+"]
    idx = int(round(n)) - 2
    if 0 <= idx < len(labels):
        return labels[idx]
    return str(n)

In [ ]:
boulder_entries = []
for i in range(n_boulders):
    bid = boulder_by_idx.get(i)
    if bid is None:
        continue
    meta = bid_to_meta.get(bid, {})
    name = meta.get("boulder_name", f"boulder_{bid}")
    crag = meta.get("crag_name", "?")
    grade = meta.get("grade_community", "?")
    grade_num = grade_to_numeric(grade)
    boulder_entries.append({
        "idx": i,
        "bid": bid,
        "name": name,
        "crag": crag,
        "grade": grade,
        "grade_num": grade_num,
        "difficulty": float(d_mean[i]),
        "difficulty_std": float(d_std[i]),
        "popularity": float(pi_mean[i]),
        "popularity_std": float(pi_std[i]),
        "ascent_count": meta.get("ascent_count", 0),
    })

valid_entries = [e for e in boulder_entries if e["grade_num"] is not None]
print(f"Boulders with grade: {len(valid_entries)} / {len(boulder_entries)}")

# Hardest Boulders

In [ ]:
Z = 1.96  # 95% credible interval

TOP_N = 30

hardest = sorted(boulder_entries,
                 key=lambda e: e["difficulty"] - Z * e["difficulty_std"],
                 reverse=True)[:TOP_N]

fig, ax = plt.subplots(figsize=(10, 0.5 * TOP_N + 1))
diffs = [e["difficulty"] for e in hardest]
errs = [Z * e["difficulty_std"] for e in hardest]
labels = [f"{e['name']}  [{e['grade']}]  {e['crag']}" for e in hardest]
colors = ["#d62728" if d > 2.0 else "#1f77b4" for d in diffs]

bars = ax.barh(range(len(labels)), diffs, xerr=errs, color=colors,
               error_kw=dict(elinewidth=1, capsize=2, alpha=0.6))
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("Inferred difficulty (posterior mean, bars = 95% CI)")
ax.set_title(f"Top {TOP_N} Hardest Boulders (ranked by lower CI bound)")

for bar, d in zip(bars, diffs):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height() / 2,
            f"{d:.2f}", va="center", fontsize=7)

fig.tight_layout()
plt.show()

# Most Popular Boulders

In [ ]:
TOP_N = 30

popular = sorted(boulder_entries,
                 key=lambda e: e["popularity"] - Z * e["popularity_std"],
                 reverse=True)[:TOP_N]

fig, ax = plt.subplots(figsize=(10, 0.5 * TOP_N + 1))
pops = [e["popularity"] for e in popular]
errs = [Z * e["popularity_std"] for e in popular]
labels = [f"{e['name']}  [{e['grade']}]  {e['crag']}" for e in popular]

ax.barh(range(len(labels)), pops, xerr=errs, color="#2ca02c",
        error_kw=dict(elinewidth=1, capsize=2, alpha=0.6))
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("Inferred popularity (posterior mean, bars = 95% CI)")
ax.set_title(f"Top {TOP_N} Most Popular Boulders (ranked by lower CI bound)")

fig.tight_layout()
plt.show()

# Most Ascended Boulders

In [ ]:
TOP_N = 30

ascended = sorted(boulder_entries, key=lambda e: e["ascent_count"], reverse=True)[:TOP_N]

fig, ax = plt.subplots(figsize=(10, 0.5 * TOP_N + 1))
counts = [e["ascent_count"] for e in ascended]
labels = [f"{e['name']}  [{e['grade']}]  {e['crag']}" for e in ascended]

ax.barh(range(len(labels)), counts, color="#ff7f0e")
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("Ascent count")
ax.set_title(f"Top {TOP_N} Most Ascended Boulders")

fig.tight_layout()
plt.show()

# Inferred Difficulty vs Community Grade

In [ ]:
valid = [e for e in boulder_entries if e["grade_num"] is not None]
grades = np.array([e["grade_num"] for e in valid])
diffs = np.array([e["difficulty"] for e in valid])
ascents = np.array([e["ascent_count"] for e in valid])

def weighted_median(values, weights):
    order = np.argsort(values)
    values = values[order]
    weights = weights[order]
    cumsum = np.cumsum(weights)
    cutoff = cumsum[-1] / 2
    return values[np.searchsorted(cumsum, cutoff)]

fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(grades, diffs, alpha=0.3, s=ascents, c="#1f77b4", edgecolors="none")

unique_grades = sorted(set(int(g) for g in grades))
medians = []
for g in unique_grades:
    mask = grades.astype(int) == g
    if mask.sum() > 0:
        medians.append((g, weighted_median(diffs[mask], ascents[mask])))
if medians:
    mg, md = zip(*medians)
    ax.plot(mg, md, "o-", color="#d62728", linewidth=2, markersize=5, label="Median (weighted)")

ax.set_xticks(sorted(set(int(g) for g in grades)))
ax.set_xticklabels([grade_label(g) for g in sorted(set(int(g) for g in grades))],
                   rotation=45, fontsize=8)
ax.set_xlabel("Community grade")
ax.set_ylabel("Inferred difficulty (posterior mean)")
ax.set_title("Inferred Difficulty vs Community Grade")
ax.legend()
fig.tight_layout()
plt.show()

# Inferred Difficulty vs Community Grade (with Uncertainty)

Marker size represents the posterior standard deviation of the boulder difficulty estimate.
Larger dots = more uncertain difficulty estimates (fewer ascents).

In [ ]:
valid = [e for e in boulder_entries if e["grade_num"] is not None]
grades_v = np.array([e["grade_num"] for e in valid])
diffs_v = np.array([e["difficulty"] for e in valid])
stds = np.array([e["difficulty_std"] for e in valid])
ascents_v = np.array([e["ascent_count"] for e in valid])

fig, ax = plt.subplots(figsize=(10, 6))

sizes = np.clip(stds * 50, 5, 300)

scatter = ax.scatter(grades_v, diffs_v, s=sizes, alpha=0.3,
                     c=stds, cmap="plasma", edgecolors="none")

unique_grades = sorted(set(int(g) for g in grades_v))
medians = []
for g in unique_grades:
    mask = grades_v.astype(int) == g
    if mask.sum() > 0:
        medians.append((g, weighted_median(diffs_v[mask], ascents_v[mask])))
if medians:
    mg, md = zip(*medians)
    ax.plot(mg, md, "o-", color="#d62728", linewidth=2,
            markersize=5, label="Median (weighted)")

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Difficulty posterior std (uncertainty)")

ax.set_xticks(sorted(set(int(g) for g in grades_v)))
ax.set_xticklabels([grade_label(g) for g in sorted(set(int(g) for g in grades_v))],
                   rotation=45, fontsize=8)
ax.set_xlabel("Community grade")
ax.set_ylabel("Inferred difficulty (posterior mean)")
ax.set_title("Inferred Difficulty vs Community Grade (size = uncertainty)")
ax.legend()
fig.tight_layout()
plt.show()

# Inferred Popularity vs Ascent Count

In [ ]:
ascents = np.array([e["ascent_count"] for e in boulder_entries])
pops = np.array([e["popularity"] for e in boulder_entries])

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(ascents, pops, alpha=0.3, s=12, c="#2ca02c", edgecolors="none")
ax.set_xlabel("Ascent count")
ax.set_ylabel("Inferred popularity (posterior mean)")
ax.set_title("Inferred Popularity vs Ascent Count")
fig.tight_layout()
plt.show()

# Boulder Difficulty vs Popularity (colored by grade)

In [ ]:
valid = [e for e in boulder_entries if e["grade_num"] is not None]
b_grades = np.array([e["grade_num"] for e in valid])
b_diffs = np.array([e["difficulty"] for e in valid])
b_pops = np.array([e["popularity"] for e in valid])

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(b_diffs, b_pops, c=b_grades, cmap="RdYlGn", alpha=0.5, s=15, edgecolors="none")
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("Community grade")
cbar.set_ticks(sorted(set(int(g) for g in b_grades)))
cbar.set_ticklabels([grade_label(g) for g in sorted(set(int(g) for g in b_grades))], fontsize=6)
ax.set_xlabel("Inferred difficulty")
ax.set_ylabel("Inferred popularity")
ax.set_title("Boulder Difficulty vs Popularity (posterior means)")
fig.tight_layout()
plt.show()

# Climber Ability vs Prolificity (colored by ascents)

In [ ]:
climber_ascents = {}
with open(ASCENTS_PATH) as f:
    for line in f:
        a = json.loads(line)
        cid = a["climber_id"]
        if cid is not None and a["ascent_type"] != "toprope":
            climber_ascents[cid] = climber_ascents.get(cid, 0) + 1

climber_to_idx = data["climber_to_idx"]
c_ascents = np.array([
    climber_ascents.get(cid, 0)
    for cid in sorted(climber_to_idx, key=climber_to_idx.get)
])

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(theta_mean, alpha_mean, c=np.log1p(c_ascents),
                alpha=0.4, s=10, edgecolors="none")
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("log(1 + ascents)")
ax.set_xlabel("Inferred ability (\u03b8, posterior mean)")
ax.set_ylabel("Inferred prolificity (\u03b1, posterior mean)")
ax.set_title("Climber Ability vs Prolificity")
fig.tight_layout()
plt.show()

In [ ]:
climber_name_map = {}
with open(ASCENTS_PATH) as f:
    for line in f:
        a = json.loads(line)
        cid = a["climber_id"]
        name = a["climber_name"]
        if cid is not None and name and name != "Anonymous" and cid not in climber_name_map:
            climber_name_map[cid] = name

climber_entries = []
for i, cid in enumerate(sorted(climber_to_idx, key=climber_to_idx.get)):
    name = climber_name_map.get(cid, f"climber_{cid}")
    climber_entries.append({
        "idx": i,
        "cid": cid,
        "name": name,
        "ability": float(theta_mean[i]),
        "ability_std": float(theta_std[i]),
        "prolificity": float(alpha_mean[i]),
        "ascents": int(c_ascents[i]),
    })

print(f"Climbers with name: {len(climber_name_map)}")

# Strongest Climbers

In [ ]:
Z = 1.96
TOP_N = 30

strongest = sorted(climber_entries,
                   key=lambda e: e["ability"] - Z * e.get("ability_std", 0),
                   reverse=True)[:TOP_N]

fig, ax = plt.subplots(figsize=(10, 0.5 * TOP_N + 1))
abilities = [e["ability"] for e in strongest]
errs = [Z * e.get("ability_std", 0) for e in strongest]
labels = [f"{e['name']}  ({e['ascents']} asc)" for e in strongest]

ax.barh(range(len(labels)), abilities[::-1], xerr=errs[::-1], color="#1f77b4",
        error_kw=dict(elinewidth=1, capsize=2, alpha=0.6))
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels[::-1], fontsize=8)
ax.set_xlabel("Inferred ability \u03b8 (posterior mean, bars = 95% CI)")
ax.set_title(f"Top {TOP_N} Strongest Climbers (ranked by lower CI bound)")

fig.tight_layout()
plt.show()

# Climber Lookup

Search by name (case-insensitive substring match).

In [ ]:
def fuzzy_find_climber(query: str, entries: list, top_k: int = 10):
    q = query.lower()
    matches = [e for e in entries if q in e["name"].lower()]
    matches.sort(key=lambda e: len(e["name"]))
    return matches[:top_k]

In [ ]:
QUERY = "Giovanni Deligios"

matches = fuzzy_find_climber(QUERY, climber_entries)
if not matches:
    print(f"No climber found matching '{QUERY}'")
else:
    print(f"Matches for '{QUERY}':")
    for e in matches:
        rank = sum(1 for x in theta_mean if x > e["ability"]) + 1
        pct = (1 - rank / n_climbers) * 100
        print(f"\n  {e['name']}")
        print(f"    ability (\u03b8):     {e['ability']:.3f} \u00b1 {e['ability_std']:.3f}")
        print(f"    prolificity (\u03b1):  {e['prolificity']:.3f}")
        print(f"    ascents:          {e['ascents']}")
        print(f"    rank:             #{rank} / {n_climbers}")
        print(f"    percentile:       {pct:.1f}")

# Predicting Grade from Boulder Elo

Weighted regression: `grade = a·elo + b·popularity + c`
Weights are inverse variance of difficulty (posterior precision), so uncertain boulders count less.

In [ ]:
POP_FILTER = -1000  # keep all boulders; uncertainty weights handle the filtering

valid = [e for e in boulder_entries if e["grade_num"] is not None and e["popularity"] >= POP_FILTER]
grd = np.array([e["grade_num"] for e in valid])
dif = np.array([e["difficulty"] for e in valid])
pop = np.array([e["popularity"] for e in valid])
diff_std_arr = np.array([e["difficulty_std"] for e in valid])
pop_std_arr = np.array([e["popularity_std"] for e in valid])

best_coef = None
for _pass in range(5):
    prec = 1.0 / (diff_std_arr**2 + 0.001)
    weights = np.clip(prec, 0, prec.mean() * 20)

    X = np.stack([dif, pop, np.ones_like(dif)], axis=1)
    W = np.diag(weights)
    coef = np.linalg.inv(X.T @ W @ X) @ X.T @ W @ grd
    b_elo, b_pop, b_int = coef[0], coef[1], coef[2]
    best_coef = coef

pred = X @ coef
residuals = grd - pred

ss_res = np.sum(weights * residuals**2)
ss_tot = np.sum(weights * (grd - np.average(grd, weights=weights))**2)
r2 = 1 - ss_res / ss_tot

pred_std = np.sqrt(b_elo**2 * diff_std_arr**2 + b_pop**2 * pop_std_arr**2)
Z = 1.96
pred_lower = pred - Z * pred_std
pred_upper = pred + Z * pred_std

print(f"grade = {b_elo:+.3f}\u00b7elo {b_pop:+.3f}\u00b7popularity + {b_int:.3f}")
print(f"Weighted R\u00b2 = {r2:.4f}")
print(f"Predicted grade std: mean={pred_std.mean():.3f}  median={np.median(pred_std):.3f}  max={pred_std.max():.3f}")

fig, ax = plt.subplots(figsize=(7, 7))
ax.errorbar(pred, grd, xerr=Z*pred_std, fmt="none", ecolor="#cccccc", alpha=0.3, linewidth=0.5)
ax.scatter(pred, grd, s=weights*2+5, alpha=0.3, c=pop, cmap="viridis", edgecolors="none")
ax.plot([grd.min(), grd.max()], [grd.min(), grd.max()], "--", color="#d62728", linewidth=1, label="y = x")
ax.set_yticks(sorted(set(int(g) for g in grd)))
ax.set_yticklabels([grade_label(g) for g in sorted(set(int(g) for g in grd))], fontsize=7)
ax.set_xticks(sorted(set(int(g) for g in grd)))
ax.set_xticklabels([grade_label(g) for g in sorted(set(int(g) for g in grd))], rotation=45, fontsize=7)
ax.set_xlabel("Predicted grade (bars = 95% CI)")
ax.set_ylabel("Actual community grade")
ax.set_title(f"Grade Prediction from Elo (weighted by posterior precision, R\u00b2={r2:.3f})")
ax.legend()
fig.tight_layout()

import csv
csv_path = "boulders_predicted_bayesian.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["name", "ascents", "grade", "popularity", "elo",
                     "elo_std", "pop_std", "inferred_grade",
                     "inferred_grade_lower", "inferred_grade_upper",
                     "inferred_grade_str", "delta"])
    for e, p, pl, pu, ps, res in zip(valid, pred, pred_lower, pred_upper, pred_std, residuals):
        writer.writerow([
            e["name"], e["ascent_count"], e["grade"],
            f"{e['popularity']:.4f}", f"{e['difficulty']:.4f}",
            f"{e['difficulty_std']:.4f}", f"{e['popularity_std']:.4f}",
            f"{p:.3f}", f"{pl:.3f}", f"{pu:.3f}",
            grade_label(p), f"{res:+.3f}"
        ])
print(f"\nSaved {len(valid)} rows to {csv_path}")
plt.show()

# Difficulty Residuals — Sandbags and Soft Touches

Boulders where inferred difficulty deviates most from the grade-based linear expectation.

In [ ]:
POP_FILTER = 1

valid = [e for e in boulder_entries if e["grade_num"] is not None and e["popularity"] >= POP_FILTER]
grades = np.array([e["grade_num"] for e in valid])
diffs = np.array([e["difficulty"] for e in valid])
pops = np.array([e["popularity"] for e in valid])

X = np.stack([grades, np.ones_like(grades)], axis=1)
coef, _, _, _ = np.linalg.lstsq(X, diffs, rcond=None)
slope, intercept = coef[0], coef[1]
predicted_diff = grades * slope + intercept
residuals = diffs - predicted_diff
print(f"Linear fit: diff = {slope:.3f} * grade + {intercept:.3f}")

outlier_n = 15
outlier_idx = np.argsort(residuals)
softest_idx = outlier_idx[:outlier_n]
sandbag_idx = outlier_idx[-outlier_n:]

fig, ax = plt.subplots(figsize=(10, 6))
sizes = np.clip(pops - pops.min(), 10, 200)
ax.scatter(grades, residuals, s=sizes, alpha=0.25, c="#7f7f7f", edgecolors="none")
ax.scatter(grades[softest_idx], residuals[softest_idx], alpha=0.9, s=sizes[softest_idx]*2,
           c="#1f77b4", edgecolors="black", linewidth=0.5, label=f"Top {outlier_n} soft (easier than grade)")
ax.scatter(grades[sandbag_idx], residuals[sandbag_idx], alpha=0.9, s=sizes[sandbag_idx]*2,
           c="#d62728", edgecolors="black", linewidth=0.5, label=f"Top {outlier_n} sandbag (harder than grade)")

for i in softest_idx:
    ax.annotate(valid[i]["name"], (grades[i], residuals[i]),
                fontsize=5, rotation=30, alpha=0.8)
for i in sandbag_idx:
    ax.annotate(valid[i]["name"], (grades[i], residuals[i]),
                fontsize=5, rotation=30, alpha=0.8)

ax.axhline(0, color="black", linewidth=0.5, linestyle="--")
ax.set_xticks(sorted(set(int(g) for g in grades)))
ax.set_xticklabels([grade_label(g) for g in sorted(set(int(g) for g in grades))],
                   rotation=45, fontsize=8)
ax.set_xlabel("Community grade")
ax.set_ylabel("Residual (positive = harder than grade, negative = easier)")
ax.set_title(f"Difficulty Residuals (size = popularity, filter pop >= {POP_FILTER})")
ax.legend(markerscale=0.5)
fig.tight_layout()
plt.show()
print("\nSofties (easier than predicted)")
for i in softest_idx:
    print(f"   * {valid[i]['name']} (grade {valid[i]['grade']})  "
          f"res={residuals[i]:+.3f}  diff={diffs[i]:.3f}  "
          f"asc={valid[i]['ascent_count']}  pop={pops[i]:.2f}")
print("\nMost Sandbagged (harder than predicted)")
for i in sandbag_idx:
    print(f"   * {valid[i]['name']} (grade {valid[i]['grade']})  "
          f"res={residuals[i]:+.3f}  diff={diffs[i]:.3f}  "
          f"asc={valid[i]['ascent_count']}  pop={pops[i]:.2f}")

# Boulder Lookup

Search by name (case-insensitive substring match).

In [ ]:
def fuzzy_find(query: str, entries: list, top_k: int = 10):
    q = query.lower()
    matches = [e for e in entries if q in e["name"].lower()]
    matches.sort(key=lambda e: len(e["name"]))
    return matches[:top_k]

In [ ]:
QUERY = "New kids"

matches = fuzzy_find(QUERY, boulder_entries)
if not matches:
    print(f"No boulder found matching '{QUERY}'")
else:
    print(f"Matches for '{QUERY}':\n")
    for e in matches:
        if e["grade_num"] is not None:
            predicted = slope * e["grade_num"] + intercept
            residual = e["difficulty"] - predicted
        else:
            residual = float("nan")
        print(f"  {e['name']}")
        print(f"    community grade:  {e['grade']}")
        print(f"    inferred diff:    {e['difficulty']:.3f} \u00b1 {e['difficulty_std']:.3f}")
        print(f"    residual:         {residual:+.3f}  (+ = sandbag, - = soft)")
        print(f"    ascents:          {e['ascent_count']}")
        print(f"    popularity:       {e['popularity']:.3f}")
        print(f"    crag:             {e['crag']}")
        print()

# Posterior Distributions — Ability & Difficulty

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(theta_mean, bins=80, color="#1f77b4", edgecolor="white", alpha=0.8)
axes[0].set_xlabel("Inferred ability (\u03b8, posterior mean)")
axes[0].set_ylabel("Number of climbers")
axes[0].set_title("Climber Ability Distribution")

axes[1].hist(d_mean, bins=80, color="#d62728", edgecolor="white", alpha=0.8)
axes[1].set_xlabel("Inferred difficulty (d, posterior mean)")
axes[1].set_ylabel("Number of boulders")
axes[1].set_title("Boulder Difficulty Distribution")

fig.tight_layout()
plt.show()

# Posterior Distributions — Global Parameters

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(beta_samples, bins=40, color="#9467bd", edgecolor="white", alpha=0.8)
axes[0].axvline(beta_mean, color="black", linestyle="--", label=f"mean = {beta_mean:.3f}")
axes[0].set_xlabel("\u03b2 (flash penalty)")
axes[0].set_ylabel("Posterior samples")
axes[0].set_title(f"Flash penalty \u03b2 = {beta_mean:.3f} \u00b1 {beta_std:.3f}")
axes[0].legend()

axes[1].hist(gamma_samples, bins=40, color="#8c564b", edgecolor="white", alpha=0.8)
axes[1].axvline(gamma_mean, color="black", linestyle="--", label=f"mean = {gamma_mean:.3f}")
axes[1].set_xlabel("\u03b3 (selectivity)")
axes[1].set_title(f"Selectivity \u03b3 = {gamma_mean:.4f}")
axes[1].legend()

axes[2].hist(mu_samples, bins=40, color="#bcbd22", edgecolor="white", alpha=0.8)
axes[2].axvline(mu_mean, color="black", linestyle="--", label=f"mean = {mu_mean:.3f}")
axes[2].set_xlabel("\u03bc (shift)")
axes[2].set_title(f"Shift \u03bc = {mu_mean:.3f} \u00b1 {mu_std:.3f}")
axes[2].legend()

fig.tight_layout()
plt.show()

# Hyperprior Distributions

In [ ]:
sigma_names = ["sigma_ability", "sigma_prolificity", "sigma_difficulty", "sigma_popularity"]
sigma_labels = ["Ability scale (\u03c3_\u03b8)", "Prolificity scale (\u03c3_\u03b1)",
                "Difficulty scale (\u03c3_d)", "Popularity scale (\u03c3_\u03c0)"]

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, name, label in zip(axes, sigma_names, sigma_labels):
    samples = idata.posterior[name].values[0]
    ax.hist(samples, bins=30, color="#17becf", edgecolor="white", alpha=0.8)
    ax.axvline(float(samples.mean()), color="black", linestyle="--")
    ax.set_xlabel(label)
    ax.set_title(f"{label} = {samples.mean():.3f}")
fig.tight_layout()
plt.show()

# Difficulty Posterior vs Ascent Count

Uncertainty shrinks with more ascents, as expected from a Bayesian model.

In [ ]:
ascents_arr = np.array([e["ascent_count"] for e in boulder_entries])
diff_std = d_std

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(ascents_arr, diff_std, alpha=0.3, s=8, c="#d62728", edgecolors="none")
ax.set_xscale("log")
ax.set_xlabel("Ascent count (log scale)")
ax.set_ylabel("Difficulty posterior std")
ax.set_title("Posterior Uncertainty vs Ascent Count")
fig.tight_layout()
plt.show()